# Preprocessing del Dataset

Questo notebook si occupa della fase di **preparazione e pulizia del dataset** di ticket aziendali.

L'obiettivo è trasformare i dati grezzi in un formato adatto all'addestramento del modello di Machine Learning.

I passaggi eseguiti sono:
1. Caricamento del dataset
2. Pulizia testuale
3. Rimozione duplicati e valori mancanti
4. Unione di oggetto e descrizione
5. Creazione della struttura finale text/label

## 1. Importazione delle librerie

Le librerie utilizzate in questa fase sono:
- **pandas**: per caricare e manipolare il dataset in formato tabellare (DataFrame)
- **re** e **unicodedata**: per la pulizia testuale a basso livello

In [ ]:
import re
import unicodedata
import pandas as pd

## 2. Funzione di pulizia del testo

Prima di caricare il dataset ho definito una funzione di supporto per standardizzare il testo.

Questa funzione applica in sequenza:
- **Normalizzazione Unicode**: gestisce caratteri accentati e problemi di encoding
- **Conversione in minuscolo**: uniforma il testo
- **Rimozione caratteri speciali**: mantiene solo lettere, numeri e vocali accentate tipiche dell'italiano
- **Compressione spazi multipli**: rimuove spazi inutili

In [ ]:
def clean_text(s: str) -> str:
    """
    Pulizia testuale applicata a ciascun campo prima dell'unione.
    """
    if s is None:
        return ""
    s = str(s)

    # Normalizzazione unicode (gestione encoding / caratteri accentati)
    s = unicodedata.normalize("NFKC", s)

    # Conversione in minuscolo
    s = s.lower()

    # Sostituisce newline/tab con spazio
    s = s.replace("\n", " ").replace("\t", " ")

    # Rimuove caratteri speciali mantenendo lettere, numeri e vocali accentate
    s = re.sub(r"[^\w\sàèéìòù]", " ", s, flags=re.UNICODE)

    # Comprime spazi multipli
    s = re.sub(r"\s+", " ", s).strip()

    return s

## 3. Caricamento del dataset

Il dataset è in formato CSV e contiene **300 righe** e **5 colonne**:
- `id` → identificativo unico del ticket
- `oggetto` → breve titolo della richiesta
- `descrizione` → testo completo inserito dall'utente
- `categoria` → classe del ticket (Amministrazione, Tecnico, Commerciale)
- `priorita` → livello di urgenza (Bassa, Media, Alta)

La funzione `pd.read_csv()` carica il file e lo trasforma in un DataFrame, una struttura tabellare simile a Excel ma gestibile direttamente in Python.

In [ ]:
df = pd.read_csv("dataset_tickets_pw18.csv")

print(f"Dimensioni dataset: {df.shape[0]} righe x {df.shape[1]} colonne")
df.head()

## 4. Pulizia del testo e gestione valori mancanti

Prima di procedere verifico lo stato del dataset:
- I valori mancanti nei campi testuali vengono sostituiti con una stringa vuota
- La funzione `clean_text` viene applicata a ciascun campo

Questa fase è fondamentale per avere un dataset coerente e senza rumore inutile.

In [ ]:
# Controllo valori mancanti prima della pulizia
print("Valori mancanti per colonna:")
print(df.isnull().sum())

In [ ]:
# Sostituzione valori mancanti e pulizia testuale
df["oggetto"] = df["oggetto"].fillna("").apply(clean_text)
df["descrizione"] = df["descrizione"].fillna("").apply(clean_text)

print("Pulizia completata.")
df[["oggetto", "descrizione"]].head(3)

## 5. Rimozione duplicati

Un ticket duplicato (stesso oggetto e stessa descrizione) non aggiunge valore al dataset e potrebbe portare il modello ad apprendere in modo distorto. Per questo motivo vengono rimossi.

In [ ]:
righe_prima = len(df)
df = df.drop_duplicates(subset=["oggetto", "descrizione"]).reset_index(drop=True)
righe_dopo = len(df)

print(f"Righe prima: {righe_prima} | Righe dopo: {righe_dopo} | Duplicati rimossi: {righe_prima - righe_dopo}")

## 6. Unione di oggetto e descrizione

La parte più importante per il modello è il testo del ticket. Ho deciso di unire `oggetto` e `descrizione` in un unico campo chiamato `testo`.

La motivazione è semplice:
- L'**oggetto** da solo è spesso troppo generico (es. "Urgente", "Problema")
- La **descrizione** da sola può essere troppo lunga e poco strutturata

Unendoli si ottiene una rappresentazione più completa che aiuta il modello a capire sia il contesto che le parole chiave.

In [ ]:
df["testo"] = (df["oggetto"] + " " + df["descrizione"]).str.strip()

# Esempio di testo risultante
print("Esempio di testo unificato:")
print(df["testo"].iloc[0])

## 7. Struttura finale del dataset

Al termine della preparazione il dataset viene ridotto a due strutture principali:
- **text** + **label categoria** → per addestrare il modello di classificazione della categoria
- **text** + **label priorità** → per addestrare il modello di stima della priorità

Questa struttura è quella richiesta dai modelli di Machine Learning supervisionato.

In [ ]:
df_cat = df[["testo", "categoria"]].rename(columns={"testo": "text", "categoria": "label"})
df_prio = df[["testo", "priorita"]].rename(columns={"testo": "text", "priorita": "label"})

print("Dataset per categoria:")
print(df_cat["label"].value_counts())
print("\nDataset per priorità:")
print(df_prio["label"].value_counts())

## Riepilogo

Il preprocessing è completato. Il dataset è ora:
- ✅ Pulito da valori mancanti e duplicati
- ✅ Normalizzato a livello testuale
- ✅ Strutturato in `text` e `label` pronti per il training

Il passo successivo è nel notebook **Training.ipynb**, dove verrà eseguita la vettorizzazione TF-IDF e l'addestramento dei modelli di classificazione.